# BAi tap: Huan luyen mo hinh dung cay quyet dinh de du doan gia cua Forex voi cac tham so dau vao nhu sau:
# - Open, High, Low, Volume, Momentum, RSI, MACD, Bollinger Bands
# - Chia du lieu thanh tap train va tap test voi ti le 80% va 20%
# - Su dung thuat toan Decision Tree de huan luyen mo hinh
# - Tinh MSE va R-squared cua mo hinh
# - In ra ket qua cua mo hinh


### Load data

In [1]:
import sys
sys.path.append("../Common/")

In [2]:
def detectSignal(symbol, from_date, to_date, timeframe):
	import CommonMT5
	from datetime import datetime, timedelta
	import MetaTrader5 as mt5
	import talib
	import redis
	from sklearn.tree import DecisionTreeRegressor
	import pandas as pd
	import numpy as np
	
	# symbol = 'XAUUSD.sml' # Symbol tren san giao dich, y chang voi symbol tren san MT5
	# from_date = (datetime.now() - timedelta(days=100)).strftime('%Y-%m-%d')  # Lấy ngày hiện tại
	# to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	# timeframe = mt5.TIMEFRAME_D1

	data = CommonMT5.CommonMT5.loaddataMT5_FromTo(symbol, from_date, to_date, timeframe)
	
	# Open, High, Low, Volume, Momentum, RSI, MACD, Bollinger Bands
	data['Momentum'] = talib.MOM(data['Close'], timeperiod=1)
	data['RSI'] = talib.RSI(data['Close'], timeperiod=14)
	data['MACD'], data['MACD_signal'], data['MACD_hist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
	data['Bollinger Bands'], data['Bollinger Bands_upper'], data['Bollinger Bands_lower'] = talib.BBANDS(data['Close'], timeperiod=20, nbdevup=2, nbdevdn=2)

	# data.dropna(inplace=True)
	data.fillna(data.mean(), inplace=True)

	# # Dua ['Low', 'Volume', 'MACD'] de huan luyen mo hinh
	X = data[['Low', 'Volume', 'MACD']]
	y = data['Close']

	model = DecisionTreeRegressor()
	model.fit(X, y)
	predictions = model.predict(X)

	data['Predicted_Close'] = predictions

	data['Buy_Signal'] = (data['Predicted_Close'] >= data['Close']) & (data['Momentum'] > 0) & (data['RSI'] < 30)
	data['Sell_Signal'] = (data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 0) & (data['RSI'] > 70)

	last_record = data.iloc[-2] 

		# ##############################################Step 3: Đẩy dữ liệu qua Redis##############################################
	# Nếu có tín hiệu thì đẩy qua Redis
	# Datetime  Open    High    Low	Close   Volume  SMA short_ema   long_ema    MACD    Signal_Line     Buy_Signal      Sell_Signal
	# Tạo kết nối Redis
	r = redis.Redis(host='localhost', port=6379, db=15)
	# Tạo hash key
	hash_key = 'Buoi16_Bai tap tai lop K13'
	   
	# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
	if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
		for field, value in last_record.to_dict().items():
			# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
			if isinstance(value, pd.Timestamp):
				value = value.isoformat()
			elif isinstance(value, (int, np.uint64)):
				value = str(value)
			r.hset(hash_key, field, value)  
			r.hset(hash_key, 'Symbol', symbol)
			r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		print(last_record)   
	else:
		print(last_record)
		print('Không có tín hiệu!')


In [3]:
# data

In [4]:
# import talib
# # Open, High, Low, Volume, Momentum, RSI, MACD, Bollinger Bands
# data['Momentum'] = talib.MOM(data['Close'], timeperiod=1)
# data['RSI'] = talib.RSI(data['Close'], timeperiod=14)
# data['MACD'], data['MACD_signal'], data['MACD_hist'] = talib.MACD(data['Close'], fastperiod=12, slowperiod=26, signalperiod=9)
# data['Bollinger Bands'], data['Bollinger Bands_upper'], data['Bollinger Bands_lower'] = talib.BBANDS(data['Close'], timeperiod=20, nbdevup=2, nbdevdn=2)

# data.dropna(inplace=True)


In [5]:
# data

# To hop cac tham so dau vao thanh mot tap du lieu

In [6]:
# import numpy as np
# from itertools import combinations

# basic_features =  ['Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI', 'MACD', 'Bollinger Bands'] # 8 bien doc lap

# # Tạo tất cả tổ hợp có thể từ basic_features
# # Sử dụng itertools để tạo tất cả các tổ hợp của các đặc trưng
# features_combinations = []
# for r in range(1, len(basic_features) + 1):
#     for combo in combinations(basic_features, r):
#         features_combinations.append(list(combo))

# print(features_combinations)
# print(len(features_combinations))

In [7]:
# import numpy as np
# from itertools import combinations
# # Import các thư viện cần thiết
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.metrics import mean_squared_error
# import matplotlib.pyplot as plt

# # Features: Open, High, Low, Volume, Momentum, RSI, MACD, Bollinger Bands
# # Target: Close
# basic_features = ['Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI', 'MACD', 'Bollinger Bands']
# y = data['Close']

# # # Tạo tất cả tổ hợp có thể từ basic_features
# # # Sử dụng itertools để tạo tất cả các tổ hợp của các đặc trưng
# # features_combinations = []
# for r in range(1, len(basic_features) + 1):
#     for combo in combinations(basic_features, r):
#         features_combinations.append(list(combo))

# # Luu trữ kết quả MSE cho từng tổ hợp
# mse_scores = {} # Dictionary luu trữ MSE cho từng tổ hợp

# # Kiểm tra từng kết hợp đặc trưng
# for features in features_combinations:
#     X = data[features].copy() # Copy dữ liệu để tránh thay đổi gốc
# 	# Thay thế NaN bằng giá trị trung bình của cột
#     X.fillna(X.mean(), inplace=True)
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
#     model = DecisionTreeRegressor(random_state=42)
#     model.fit(X_train, y_train)
    
#     predictions = model.predict(X_test)
#     mse = mean_squared_error(y_test, predictions) # Tinh MSE
    
#     mse_scores[str(features)] = mse # Them vao dictionary mse_scores

# print(mse_scores)
# # Tìm kết hợp đặc trưng với MSE thấp nhất
# best_features = min(mse_scores, key=mse_scores.get)
# print(f"Best features: {best_features} with MSE: {mse_scores[best_features]}")

In [8]:
# # Dua ['Low', 'Volume', 'MACD'] de huan luyen mo hinh
# X = data[['Low', 'Volume', 'MACD']]
# y = data['Close']

# model = DecisionTreeRegressor()
# model.fit(X, y)
# predictions = model.predict(X)

# data['Predicted_Close'] = predictions

# data['Buy_Signal'] = (data['Predicted_Close'] >= data['Close']) & (data['Momentum'] > 0) & (data['RSI'] < 30)
# data['Sell_Signal'] = (data['Predicted_Close'] < data['Close']) & (data['Momentum'] < 0) & (data['RSI'] > 70)

# data

# Dua vao mo hinh do de lam bai tap OG

# Shedule

In [ ]:
# 1 ngay chay 1 lan vao luc 21:35:00
import schedule
import time
import MetaTrader5 as mt5
from datetime import datetime, timedelta

def schedule_detectSignal():
	symbol = 'XAUUSD.sml'
	from_date = (datetime.now() - timedelta(days=40)).strftime('%Y-%m-%d')  # Lấy ngày hôm qua
	to_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
	detectSignal(symbol, from_date, to_date, mt5.TIMEFRAME_D1)

schedule.every(1).day.at("21:50:00").do(schedule_detectSignal)	

while True:
	schedule.run_pending()
	time.sleep(1)


Datetime                 2025-11-06 00:00:00
Open                                3968.765
High                                4019.575
Low                                  3964.53
Close                               3977.235
Volume                                359440
Momentum                               -1.99
RSI                                50.923509
MACD                                     NaN
MACD_signal                              NaN
MACD_hist                                NaN
Bollinger Bands                  4329.337107
Bollinger Bands_upper               4082.936
Bollinger Bands_lower            3836.534893
Predicted_Close                     3977.235
Buy_Signal                             False
Sell_Signal                            False
Name: 28, dtype: object
Không có tín hiệu!
